In [1]:
import re
import json
import pandas as pd
import os
from pathlib import Path

In [2]:
date = "16APR2026"

In [3]:


def extract_snapshots(log_file_path):
    pattern = re.compile(r'WINDOW_SNAPSHOT:\s*(\{.*\})')

    snapshots = []

    with open(log_file_path, 'r') as f:
        for line in f:
            match = pattern.search(line)
            if match:
                try:
                    data = json.loads(match.group(1))
                    snapshots.append(data)
                except json.JSONDecodeError:
                    continue

    return snapshots


def snapshots_to_dataframe(snapshots):
    df = pd.DataFrame(snapshots)

    # Optional: convert time column
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])

    return df


In [4]:
# ---- Usage ----
log_path = Path(f"assets/logs/{date}/streamer.log")


snapshots = extract_snapshots(log_path)
df = snapshots_to_dataframe(snapshots)

In [5]:
df.shape

(765, 17)

In [6]:
df.head()

,time,nifty,atm,ATM-3_CE,ATM-3_PE,ATM-2_CE,ATM-2_PE,ATM-1_CE,ATM-1_PE,ATM_CE,ATM_PE,ATM+1_CE,ATM+1_PE,ATM+2_CE,ATM+2_PE,ATM+3_CE,ATM+3_PE
0,2026-04-16 09:10:05.247,24385.20,24400,NIFTY26JUN24250CE,NIFTY26JUN24250PE,NIFTY26JUN24300CE,NIFTY26JUN24300PE,NIFTY26JUN24350CE,NIFTY26JUN24350PE,NIFTY26JUN24400CE,NIFTY26JUN24400PE,NIFTY26JUN24450CE,NIFTY26JUN24450PE,NIFTY26JUN24500CE,NIFTY26JUN24500PE,NIFTY26JUN24550CE,NIFTY26JUN24550PE
1,2026-04-16 09:15:00.993,24367.90,24350,NIFTY26JUN24200CE,NIFTY26JUN24200PE,NIFTY26JUN24250CE,NIFTY26JUN24250PE,NIFTY26JUN24300CE,NIFTY26JUN24300PE,NIFTY26JUN24350CE,NIFTY26JUN24350PE,NIFTY26JUN24400CE,NIFTY26JUN24400PE,NIFTY26JUN24450CE,NIFTY26JUN24450PE,NIFTY26JUN24500CE,NIFTY26JUN24500PE
2,2026-04-16 09:15:01.495,24381.50,24400,NIFTY26JUN24250CE,NIFTY26JUN24250PE,NIFTY26JUN24300CE,NIFTY26JUN24300PE,NIFTY26JUN24350CE,NIFTY26JUN24350PE,NIFTY26JUN24400CE,NIFTY26JUN24400PE,NIFTY26JUN24450CE,NIFTY26JUN24450PE,NIFTY26JUN24500CE,NIFTY26JUN24500PE,NIFTY26JUN24550CE,NIFTY26JUN24550PE
3,2026-04-16 09:15:03.993,24366.55,24350,NIFTY26JUN24200CE,NIFTY26JUN24200PE,NIFTY26JUN24250CE,NIFTY26JUN24250PE,NIFTY26JUN24300CE,NIFTY26JUN24300PE,NIFTY26JUN24350CE,NIFTY26JUN24350PE,NIFTY26JUN24400CE,NIFTY26JUN24400PE,NIFTY26JUN24450CE,NIFTY26JUN24450PE,NIFTY26JUN24500CE,NIFTY26JUN24500PE
4,2026-04-16 09:17:31.492,24375.10,24400,NIFTY26JUN24250CE,NIFTY26JUN24250PE,NIFTY26JUN24300CE,NIFTY26JUN24300PE,NIFTY26JUN24350CE,NIFTY26JUN24350PE,NIFTY26JUN24400CE,NIFTY26JUN24400PE,NIFTY26JUN24450CE,NIFTY26JUN24450PE,NIFTY26JUN24500CE,NIFTY26JUN24500PE,NIFTY26JUN24550CE,NIFTY26JUN24550PE


In [7]:
# df.to_excel("option_chain.xlsx")

In [8]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill
import random

# --- Config ---
COLOR_POOL = [
    "FFC7CE", "C6EFCE", "FFEB9C", "BDD7EE", "D9D2E9",
    "FCE4D6", "E2EFDA", "FFF2CC", "DDEBF7", "EAD1DC",

    "F4CCCC", "D9EAD3", "FFF2CC", "CFE2F3", "D9D2E9",
    "FCE5CD", "EAD1DC", "D0E0E3", "F9CB9C", "C9DAF8",

    "EA9999", "B6D7A8", "FFE599", "9FC5E8", "B4A7D6",
    "F6B26B", "D5A6BD", "A2C4C9", "FFD966", "A4C2F4",

    "E06666", "93C47D", "FFD966", "6FA8DC", "8E7CC3",
    "F6B26B", "C27BA0", "76A5AF", "F1C232", "6D9EEB",

    "CC0000", "6AA84F", "F1C232", "3D85C6", "674EA7",
    "E69138", "A64D79", "45818E", "BF9000", "3C78D8",

    "990000", "38761D", "BF9000", "134F5C", "351C75",
    "783F04"
]

# --- Persistent mapping ---
symbol_color_map = {}

def get_color(symbol):
    if symbol not in symbol_color_map:
        # deterministic or random
        color = COLOR_POOL[len(symbol_color_map) % len(COLOR_POOL)]
        symbol_color_map[symbol] = color
    return symbol_color_map[symbol]


def write_colored_excel(df, file_name="output.xlsx"):
    wb = Workbook()
    ws = wb.active

    # Write header
    ws.append(list(df.columns))

    for row_idx, row in df.iterrows():
        excel_row = []

        for col in df.columns:
            value = row[col]
            excel_row.append(value)

        ws.append(excel_row)

        # Apply colors AFTER writing row
        for col_idx, col in enumerate(df.columns, start=1):
            value = row[col]

            if isinstance(value, str) and value.startswith("NIFTY"):
                color = get_color(value)

                fill = PatternFill(
                    start_color=color,
                    end_color=color,
                    fill_type="solid"
                )

                ws.cell(row=row_idx + 2, column=col_idx).fill = fill

    wb.save(file_name)

In [9]:
path_ = Path(f"assets/logs/{date}/extracted_symbols/option_chain_colored.xlsx")
path_.parent.mkdir(parents=True, exist_ok=True)
write_colored_excel(df, file_name=path_)

In [27]:
def export_colored_excel(df, file_path="output.xlsx"):
    from openpyxl import Workbook
    from openpyxl.styles import PatternFill
    import random

    wb = Workbook()
    ws = wb.active
    ws.title = "Data"

    # -----------------------
    # Write header
    # -----------------------
    headers = list(df.columns)
    ws.append(headers)

    # -----------------------
    # Generate colors per column (except time)
    # -----------------------
    value_columns = [col for col in df.columns if col != "time"]

    def random_color():
        return ''.join([format(random.randint(0, 255), '02X') for _ in range(3)])

    color_map = {col: random_color() for col in value_columns}

    # -----------------------
    # Write data + apply colors per column
    # -----------------------
    for _, row in df.iterrows():
        ws.append(list(row))
        current_row = ws.max_row

        for col_idx, col_name in enumerate(headers, start=1):
            if col_name in color_map:
                fill = PatternFill(start_color=color_map[col_name],
                                   end_color=color_map[col_name],
                                   fill_type="solid")
                ws.cell(row=current_row, column=col_idx).fill = fill

    # -----------------------
    # Save file
    # -----------------------
    wb.save(file_path)

In [28]:
export_colored_excel(df, "dev/option_chain_colored.xlsx")

In [24]:
df.columns

Index(['time', 'nifty', 'atm', 'ATM-3_CE', 'ATM-3_PE', 'ATM-2_CE', 'ATM-2_PE',
       'ATM-1_CE', 'ATM-1_PE', 'ATM_CE', 'ATM_PE', 'ATM+1_CE', 'ATM+1_PE',
       'ATM+2_CE', 'ATM+2_PE', 'ATM+3_CE', 'ATM+3_PE'],
      dtype='str')

In [23]:
df

,time,nifty,atm,ATM-3_CE,ATM-3_PE,ATM-2_CE,ATM-2_PE,ATM-1_CE,ATM-1_PE,ATM_CE,ATM_PE,ATM+1_CE,ATM+1_PE,ATM+2_CE,ATM+2_PE,ATM+3_CE,ATM+3_PE
0,2026-04-15 12:14:09.362,24226.60,24250,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE,NIFTY2642124400CE,NIFTY2642124400PE
1,2026-04-15 12:14:14.611,24224.95,24200,NIFTY2642124050CE,NIFTY2642124050PE,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE
2,2026-04-15 12:14:14.861,24225.60,24250,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE,NIFTY2642124400CE,NIFTY2642124400PE
3,2026-04-15 12:17:50.863,24224.70,24200,NIFTY2642124050CE,NIFTY2642124050PE,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE
4,2026-04-15 12:17:51.613,24226.55,24250,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE,NIFTY2642124400CE,NIFTY2642124400PE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,2026-04-15 15:25:27.841,24226.80,24250,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE,NIFTY2642124400CE,NIFTY2642124400PE
449,2026-04-15 15:25:29.842,24224.25,24200,NIFTY2642124050CE,NIFTY2642124050PE,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE
450,2026-04-15 15:25:30.591,24226.70,24250,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE,NIFTY2642124400CE,NIFTY2642124400PE
451,2026-04-15 15:25:35.091,24221.80,24200,NIFTY2642124050CE,NIFTY2642124050PE,NIFTY2642124100CE,NIFTY2642124100PE,NIFTY2642124150CE,NIFTY2642124150PE,NIFTY2642124200CE,NIFTY2642124200PE,NIFTY2642124250CE,NIFTY2642124250PE,NIFTY2642124300CE,NIFTY2642124300PE,NIFTY2642124350CE,NIFTY2642124350PE
